## Tensorflow

### Datos, modelo, entrenamiento

In [1]:
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

# Si solo ejecutas esta parte (sin Parte A), definimos corpus y helpers aquí.
try:
    SEQ
except NameError:
    CORPUS = r'''
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distancia_origen(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

for i in range(10):
    print(i, fibonacci(i))
'''
    chars = sorted(set(CORPUS))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    VOCAB_SIZE = len(chars)

    def encode(s):
        return [stoi[c] for c in s]

    def decode(ids):
        return "".join(itos[i] for i in ids)

    SEQ = np.array(encode(CORPUS), dtype=np.int64)

block_size_tf = 32
X_rows_tf, Y_rows_tf = [], []
for i in range(0, len(SEQ) - block_size_tf):
    X_rows_tf.append(SEQ[i : i + block_size_tf])
    Y_rows_tf.append(SEQ[i + 1 : i + 1 + block_size_tf])

X_tf = np.stack(X_rows_tf)
Y_tf = np.stack(Y_rows_tf)

embed_dim_tf = 48
hidden_tf = 64

model_tf = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(block_size_tf,)),
        tf.keras.layers.Embedding(VOCAB_SIZE, embed_dim_tf),
        tf.keras.layers.SimpleRNN(hidden_tf, activation="tanh", return_sequences=True),
        tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(VOCAB_SIZE)),
    ]
)

model_tf.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

model_tf.summary()

history = model_tf.fit(
    X_tf,
    Y_tf,
    epochs=240,
    batch_size=16,
    verbose=0,
)
print("épocas:", len(history.history["loss"]))
print("pérdida inicial:", float(history.history["loss"][0]))
print("pérdida final:", float(history.history["loss"][-1]))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 32, 48)         │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32, 64)         │         7,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 32, 36)         │         2,340 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,300 (44.14 KB)

 Trainable params: 11,300 (44.14 KB)

 Non-trainable params: 0 (0.00 B)

épocas: 240
pérdida inicial: 3.1191232204437256
pérdida final: 0.10107297450304031


### Generación (autocompletado) con el modelo TensorFlow

In [2]:
import tensorflow as tf
import numpy as np


def complete_tf(prompt, max_new_tokens=120, temperature=0.8):
    """Usa forward de ventana fija: últimos block_size_tf caracteres."""
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        x = np.array(ids[-block_size_tf:], dtype=np.int64)
        if x.shape[0] < block_size_tf:
            pad = np.full(block_size_tf - x.shape[0], ids[0], dtype=np.int64)
            x = np.concatenate([pad, x])
        x = x.reshape(1, block_size_tf)
        logits = model_tf(x, training=False).numpy()[0, -1, :]
        logits = logits / max(temperature, 1e-6)
        logits = logits - logits.max()
        probs = np.exp(logits)
        probs = probs / probs.sum()
        next_id = int(np.random.choice(len(probs), p=probs))
        ids.append(next_id)
    return decode(ids)


print(complete_tf("def fac", max_new_tokens=90, temperature=0.1)[:450])

def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

def fibonacci(
